In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
import time
import copy
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import balanced_accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, f1_score

## Data Augmentation
La pipeline (ossia la sequenza di operazioni effettuate su un'immagine) è stata fondata sulle esigenze della rete ResNet (Immagini in ingresso 224x224 pixel su 3 canali (RGB)). Le trasformazioni attuate sono 2:

train_basic (Augmentation Leggera): Dedicata ai campioni utilizzati durante il training. Ridimensiona l'immagine e applica solo una specchiatura orizzontale casuale.

val (Validazione): Non applica alcuna alterazione artificiale. Si limita a ridimensionare e ritagliare l'immagine al centro per adattare i dati ale esigenze della rete (resize + crop + normalize).

Tali trasformazioni sono applicate in modo casuale al fine di fornire una varietà di dati a disposizione il più grande possibile.

In [ ]:
class StandardDatasetWrapper(Dataset):
    """Wrapper semplice per applicare la stessa trasformazione a tutto il subset."""
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


def get_uniform_dataloaders(data_dir, batch_size=32, val_size=0.2, random_seed=42):
    """
    Carica i dati applicando la STESSA data augmentation a tutte le classi.
    Eliminate completamente le alterazioni di luminosità e contrasto per rimuovere
    il rumore artificiale che creava falsi positivi sulla classe metallo.
    Preservate solo le variazioni geometriche pulite e la saturazione/tinta per i trasparenti.
    """
    data_transforms = {
        'train': transforms.Compose([
            transforms.Resize(256),
            # Invarianza di scala e ritaglio geometrico pulito
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
            # Rotazioni ed effetto "specchiato"
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(15),
            # Brightness e Contrast azzerati (valore = 0 significa nessuna variazione).
            # Mantenute saturazione e tinta per evitare che vetro e plastica si ancorino
            # cromaticamente allo sfondo fisso delle foto.
            transforms.ColorJitter(
                brightness=0.0,
                contrast=0.0,
                saturation=0.3,
                hue=0.05
            ),
            transforms.ToTensor(),
            # Normalizzazione standard per modelli ImageNet
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]),
        'val': transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]),
    }
    # Caricamento del dataset di base
    full_dataset = datasets.ImageFolder(data_dir)
    targets = torch.tensor(full_dataset.targets)
    class_names = full_dataset.classes
    # Split Train e Val stratificato per mantenere le proporzioni
    train_idx, val_idx = train_test_split(
        np.arange(len(targets)),
        test_size=val_size,
        random_state=random_seed,
        stratify=targets
    )
    train_subset = torch.utils.data.Subset(full_dataset, train_idx)
    val_subset = torch.utils.data.Subset(full_dataset, val_idx)
    train_dataset = StandardDatasetWrapper(train_subset, transform=data_transforms['train'])
    val_dataset = StandardDatasetWrapper(val_subset, transform=data_transforms['val'])
    # Configurazione dei Dataloader ottimizzati per GPU
    dataloaders = {
        'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True),
        'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True),
    }
    print(f"\nDataloader in assetto 'Clean' pronti per il test di controllo.")
    print(f"Rimossi: Variazioni di luminosità, contrasto, sfocatura e distorsione prospettica.")
    return dataloaders, class_names

## Addestramento del modello
Operazioni di addestramento necessarie, le funzioni ricevono in ingresso gli iperparametri e forniscono in uscita le metriche di performance ottenute.

Come primo step si costruisce la funzione che consente di inizializzare la rete: riceve in ingresso gli iperparametri e procede a caricare il modello, sbloccando solo il numero di layer passati in ingresso.

In [ ]:
def initial_reset(num_classes, training_layers=8, dropout_rate=0.2):
  """
  Inizializza EfficientNet-B0 pre-addestrata con sblocco progressivo DINAMICO.
  training_layers indica quanti blocchi sbloccare a partire dal fondo della rete (0-8).
  """
  weights = EfficientNet_B0_Weights.DEFAULT
  model = efficientnet_b0(weights=weights)
  # Inizialmente si congela l'intera rete
  for param in model.parameters():
    param.requires_grad = False
  # Si Sostituisce e si sblocca il modulo Classifier (Testa)
  n_features = model.classifier[1].in_features
  model.classifier = nn.Sequential(
      nn.Dropout(p=dropout_rate, inplace=True),
      nn.Linear(n_features, num_classes)
  )
  for param in model.classifier.parameters():
    param.requires_grad = True

  if training_layers > 0:
      available_layers = list(model.features.children())[::-1] #si estraggono i blocchi e si capovolgono
      unlocked_layers = min(training_layers, len(available_layers)) #si prende il numero di blocchi da sbloccare (tutti se un numero >8)
      for i in range(unlocked_layers):
        for param in available_layers[i].parameters():
          param.requires_grad = True #si sbloccano i layer

  return model

Metodo di utility per sviluppare i gradici relativi alla loss e all'accuratezza

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(history):
    """
    Prende in ingresso il dizionario 'history' generato da train_model
    e plotta i grafici di Loss e Accuratezza.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    #Figura con due sotto-grafici (1 riga, 2 colonne)
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    # Grafico per la Loss (Errore)
    ax1.plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    ax1.set_title('Andamento della Loss')
    ax1.set_xlabel('Epoche')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    # Grafico per l'Accuratezza
    ax2.plot(epochs, history['train_acc'], 'b-', label='Training Accuracy')
    ax2.plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy')
    ax2.set_title('Andamento dell\'Accuratezza')
    ax2.set_xlabel('Epoche')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    #Grafico per la Balanced Accuracy
    ax3.plot(epochs, history['train_bal_acc'], 'b-', label='Training Bal. Acc.')
    ax3.plot(epochs, history['val_bal_acc'], 'r-', label='Validation Bal. Acc.')
    ax3.set_title('Andamento della Balanced Accuracy')
    ax3.set_xlabel('Epoche')
    ax3.set_ylabel('Balanced Accuracy')
    ax3.legend()
    ax3.grid(True)

    plt.tight_layout()
    plt.show()

Early stopping e addestramento:

In [ ]:
class EarlyStopping:
    """Interrompe l'addestramento se la validation loss non migliora dopo un tot di epoche."""
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience     # Quante epoche aspettare prima di fermarsi
        self.min_delta = min_delta   # Miglioramento minimo richiesto per non resettare il contatore
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f"-> EarlyStopping Counter: {self.counter} su {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def train_model_advanced(model, dataloaders, optimizer, epochs, scheduler=None, class_weights=None, patience=5,label_smoothing=0.1):
    """Ciclo di addestramento avanzato con supporto a Early Stopping e ReduceLROnPlateau."""
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    if class_weights is not None:
        class_weights = class_weights.to(device)
        loss_function = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
    else:
        loss_function = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    best_val_loss = float('inf')
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'train_bal_acc':[], 'val_bal_acc':[] }
    best_model = copy.deepcopy(model.state_dict())

    # Inizializziamo il controllore dell'Early Stopping
    early_stopper = EarlyStopping(patience=patience, min_delta=0.001)

    for epoch in range(epochs):
        print(f'\nEpoca {epoch+1}/{epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            curr_loss = 0.0
            curr_correct = 0
            all_preds = []
            all_labels = []

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = loss_function(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                curr_loss += loss.item() * inputs.size(0)
                curr_correct += torch.sum(preds == labels.data)

                # Vettori estratti correttamente per scikit-learn
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            epoch_loss = curr_loss / len(dataloaders[phase].dataset)
            epoch_acc = curr_correct.double() / len(dataloaders[phase].dataset)
            epoch_bal_acc = balanced_accuracy_score(all_labels, all_preds)

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Bal. Acc: {epoch_bal_acc:.4f}')

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())
            history[f'{phase}_bal_acc'].append(epoch_bal_acc)

            # Logiche di controllo eseguite SOLO a fine validazione
            if phase == 'val':
                # Salva i pesi se è la loss minore mai vista
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_model = copy.deepcopy(model.state_dict())

                # Aggiorna lo scheduler (passando la validation loss se è un ReduceLROnPlateau)
                if scheduler is not None:
                    old_lr = optimizer.param_groups[0]['lr']
                    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                        scheduler.step(epoch_loss)
                    else:
                        scheduler.step()
                    new_lr = optimizer.param_groups[0]['lr']
                    if new_lr < old_lr:
                        print(f"Learning Rate decresciuto automaticamente a: {new_lr}")

                # Interroga l'Early Stopping
                early_stopper(epoch_loss)

        # Interrompe l'addestramento se la loss ha smesso di scendere
        if early_stopper.early_stop:
            print(f"Early Stopping attivato! Addestramento interrotto anticipatamente.")
            break

    print(f'\nAddestramento concluso. Ripristino dei pesi migliori (Miglior Validation Loss: {best_val_loss:.4f})')
    model.load_state_dict(best_model)
    return model, history

## Setup Ambiente e Gestione del Dataset

In [ ]:
from google.colab import drive

import os

# 1. Mount del Drive e posizionamento nella cartella del progetto
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/2026_MLinf_gr16/Data/')

# 2. Copia e scompattamento del dataset nello spazio locale temporaneo di Colab
!cp "./waste_type_identification.zip" "/content/"
!unzip -q "/content/waste_type_identification.zip" -d "/content/dataset/"

print("Dataset caricato e scompattato localmente con successo.")

Mounted at /content/drive
Dataset caricato e scompattato localmente con successo.


### Definizione iperparametri globali

In [ ]:
DATA_DIR = '/content/dataset/waste_type_identification'
BATCH_SIZE = 32
NUM_CLASSES = 8
NUM_EPOCHS = 40
PATIENCE = 5

LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.0001
DROPOUT_RATE = 0.2
TRAINING_LAYERS = 5
LABEL_SMOOTHING = 0.1

**Inizializzazione dei dataloaders**:  Si invoca la funzione get_uniform_dataloaders per generare i dizionari di addestramento e validazione strutturati con il campionamento pesato.

In [ ]:
# Generazione dei Dataloaders e recupero dei nomi delle classi
dataloaders, class_names = get_uniform_dataloaders(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    val_size=0.2,
    random_seed=42
)


Dataloader in assetto 'Clean' pronti per il test di controllo.
Rimossi: Variazioni di luminosità, contrasto, sfocatura e distorsione prospettica.


**Calcolo dei pesi per la loss**: Visto che il metodo di addestramento accetta un parametro opzionale associato ai pesi delle classi, si estrae la distribuzione reale dei target di addestramento per bilanciare il calcolo della Cross Entropy.

In [ ]:
# Estrazione delle etichette reali del subset di training per calcolare i pesi della loss
train_subset = dataloaders['train'].dataset.subset
train_targets = [train_subset.dataset.targets[i] for i in train_subset.indices]

# Calcolo delle frequenze e inversione dei pesi
class_counts = np.bincount(train_targets, minlength=NUM_CLASSES)
class_weights = 1.0 / class_counts.astype(float)
class_weights_normalization = class_weights / class_weights.sum()

# Conversione in tensore PyTorch
loss_weights_tensor = torch.FloatTensor(class_weights_normalization)
print(f"Pesi calcolati per il bilanciamento della Loss: {loss_weights_tensor}")

Pesi calcolati per il bilanciamento della Loss: tensor([0.1488, 0.0193, 0.0699, 0.1829, 0.1427, 0.0724, 0.1625, 0.2016])


**Inizializzazione della rete, ottimizzatore e scheduler** : si prepara l'architettura delle rete e si configura il meccanismo di decadimento del learning rate.

In [ ]:
# Inizializzazione della rete
model = initial_reset(
    num_classes=NUM_CLASSES,
    training_layers=TRAINING_LAYERS,
    dropout_rate=DROPOUT_RATE
)

#Ottimizzatore Adam configurato unicamente sui parametri attivi (requires_grad=True)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# SCHEDULER ADATTIVO:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.1)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 160MB/s]


## Addestramento e Visualizzazione delle Metriche

In [ ]:
model_addestrato, history = train_model_advanced(
    model=model,
    dataloaders=dataloaders,
    optimizer=optimizer,
    epochs=NUM_EPOCHS,
    scheduler=scheduler,
    class_weights=loss_weights_tensor,
    patience=PATIENCE,
    label_smoothing=LABEL_SMOOTHING
)

# Generazione dei grafici a 3 pannelli
plot_metrics(history)

Valutazione avanzata e matrice di confusione

In [ ]:

print("Avvio della valutazione finale sul Validation Set...")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Modello in modalità di inferenza
model_addestrato.eval()

all_preds = []
all_labels = []

# Disabilitazione del calcolo del gradiente
with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        # Trasferimento dati verso il dispositivo corretto (GPU/CPU)
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Passo di forward
        outputs = model_addestrato(inputs)
        _, preds = torch.max(outputs, 1) # Estrazione indice della classe a probabilità massima

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Conversione liste in array
y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# Calcolo e stampa delle metriche globali
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n================ METRICHE GLOBALI CONSOLIDATE ================")
print(f"Accuracy Globale: {acc:.4f}")
print(f"Precision (Macro): {prec:.4f}")
print(f"F1-Score (Macro):  {f1:.4f}")
print("==============================================================\n")

#Report testuale analitico suddiviso per singola classe di rifiuto
print("Report dettagliato per categoria merceologica:")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

# Calcolo della matrice di confusione
cm = confusion_matrix(y_true, y_pred)


plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.ylabel('Classe Reale (Ground Truth)')
plt.xlabel('Classe Predetta dal Modello')
plt.title('Matrice di Confusione Consolidata - Assetto Avanzato')
plt.tight_layout()
plt.show()

## Salvataggio finale

In [ ]:
percorso_salvataggio = '/content/drive/MyDrive/2026_MLinf_gr16/TrainingSession/efficientnet_optimized.pth'
torch.save(model_addestrato, percorso_salvataggio)

print(f"Sessione completata con successo! Modello esportato in: {percorso_salvataggio}")